In [ ]:
import xarray as xr
import numpy as np
import os

# =====================================================
# CONFIGURATION
# =====================================================
cleaned_files = [
    r"E:\ML Project\WOSC Project\EarthFormer\Cleaned_Netcdf\cleaned_1_Amphan.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Cleaned_Netcdf\cleaned_2_Asani.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Cleaned_Netcdf\cleaned_3_Dana.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Cleaned_Netcdf\cleaned_4_Fani.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Cleaned_Netcdf\cleaned_5_Gulab.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Cleaned_Netcdf\cleaned_6_Hudhud.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Cleaned_Netcdf\cleaned_7_Phailin.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Cleaned_Netcdf\cleaned_8_Titli.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Cleaned_Netcdf\cleaned_9_Yaas.nc"
]

cyclone_windows = {
    1: {"pre": ("2020-05-11","2020-05-15"), "during": ("2020-05-16","2020-05-20")},
    2: {"pre": ("2022-05-02","2022-05-06"), "during": ("2022-05-07","2022-05-11")},
    3: {"pre": ("2024-10-17","2024-10-21"), "during": ("2024-10-22","2024-10-26")},
    4: {"pre": ("2019-04-21","2019-04-25"), "during": ("2019-04-26","2019-04-30")},
    5: {"pre": ("2021-09-19","2021-09-23"), "during": ("2021-09-24","2021-09-28")},
    6: {"pre": ("2014-10-02","2014-10-06"), "during": ("2014-10-07","2014-10-11")},
    7: {"pre": ("2013-10-03","2013-10-07"), "during": ("2013-10-08","2013-10-12")},
    8: {"pre": ("2018-10-03","2018-10-07"), "during": ("2018-10-08","2018-10-12")},
    9: {"pre": ("2021-05-18","2021-05-22"), "during": ("2021-05-23","2021-05-27")},
}

input_vars  = ["to", "so", "ugo", "vgo", "zo"] 
output_vars = ["to", "so", "zo"]                 
LAT_MIN, LAT_MAX = 18.3, 19.3
LON_MIN, LON_MAX = 85.32, 86.32

DEPTH_FULL = np.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65,
                       70, 80, 90, 100, 125, 150, 175, 200, 225, 250, 275,
                       300, 350, 400, 450, 500, 550, 600, 700], dtype=np.float32)
DEPTH_MAX = 100
depth_mask = DEPTH_FULL <= DEPTH_MAX
depth_sel = DEPTH_FULL[depth_mask]

forcing_sequences = {
    "Amphan":  np.array([1.2016,2.4749,3.7246,3.5185,2.3082], dtype=np.float32), 
    "Asani":   np.array([0.5430,1.2265,1.8053,2.6470,2.7482], dtype=np.float32),
    "Dana":    np.array([0.4796,1.0196,1.6310,1.6653,0.6909], dtype=np.float32),
    "Fani":    np.array([0.5020,0.8537,0.9681,1.5638,3.0457], dtype=np.float32),
    "Gulab":   np.array([0.3706,0.7704,1.8112,2.4135,4.1904], dtype=np.float32),
    "Hudhud":  np.array([0.3107,0.6735,1.2910,2.0846,3.7276], dtype=np.float32),
    "Phailin": np.array([0.2641,0.5558,1.6974,2.8673,4.3486], dtype=np.float32),
    "Titli":   np.array([0.7031,1.4289,2.9383,3.9187,1.1154], dtype=np.float32),
    "Yaas":    np.array([0.6092,1.0683,1.8339,2.5572,1.3598], dtype=np.float32),
}
wind_vals = np.array(list(forcing_sequences.values()))
wind_mean = wind_vals.mean()
wind_std = wind_vals.std()

OUTPUT_DIR = r"E:\ML Project\WOSC Project\WOSC Dynamic\Earthformer_Arrays_Dynamic"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================
# HELPER FUNCTIONS
# =====================================================
def split_pre_during(ds, window):
    pre = ds.sel(time=slice(*window["pre"]))
    during = ds.sel(time=slice(*window["during"]))
    return pre, during

def compute_anomaly(pre, target):
    return target - pre.mean(dim="time")

def crop_chilika(X, Y):
    H, W = X.shape[2], X.shape[3]
    lat_full = np.linspace(15.0625, 22.4375, H)
    lon_full = np.linspace(82.5625, 89.9375, W)
    lat_idx = np.where((lat_full >= LAT_MIN) & (lat_full <= LAT_MAX))[0]
    lon_idx = np.where((lon_full >= LON_MIN) & (lon_full <= LON_MAX))[0]
    Xc = X[:, :, lat_idx[:, None], lon_idx, :]
    Yc = Y[:, :, lat_idx[:, None], lon_idx, :]
    if Xc.shape[2] != 8 or Xc.shape[3] != 8:
        raise ValueError(f"Chilika crop failed: {Xc.shape}")
    return Xc, Yc

def make_depth_channels(T, H, W, depth_vals):
    z = (depth_vals - depth_vals.mean()) / depth_vals.std()
    zmap = z[None, None, None, :]
    return np.tile(zmap, (T, H, W, 1))

def flatten_with_depth(A, depth_vals):
    T, D, H, W, V = A.shape
    A = np.transpose(A, (0, 2, 3, 1, 4))  # (T,H,W,D,V)
    A = A.reshape(T, H, W, D*V)
    depth_ch = make_depth_channels(T, H, W, depth_vals)
    return np.concatenate([A, depth_ch], axis=-1)

# =====================================================
# COMPUTING GLOBAL MEAN & STD
# =====================================================
print("\nGlobal scalers")

global_sum = {v: 0.0 for v in input_vars}
global_sqsum = {v: 0.0 for v in input_vars}
global_count = {v: 0 for v in input_vars}

for i, file in enumerate(cleaned_files, start=1):
    ds = xr.open_dataset(file)
    window = cyclone_windows[i]
    pre, during = split_pre_during(ds, window)

    # Using raw physical values (pre + during)
    combined = xr.concat([pre, during], dim="time")

    for v in input_vars:
        arr = combined[v].values.astype(np.float64)

        # Depth selection
        arr = arr[:, depth_mask, :, :]

        # Crop Chilika
        H, W = arr.shape[2], arr.shape[3]
        lat_full = np.linspace(15.0625, 22.4375, H)
        lon_full = np.linspace(82.5625, 89.9375, W)
        lat_idx = np.where((lat_full >= LAT_MIN) & (lat_full <= LAT_MAX))[0]
        lon_idx = np.where((lon_full >= LON_MIN) & (lon_full <= LON_MAX))[0]
        arr = arr[:, :, lat_idx[:, None], lon_idx]

        global_sum[v] += arr.sum()
        global_sqsum[v] += (arr ** 2).sum()
        global_count[v] += arr.size

# Computing final scalers
global_scalers = {}

for v in input_vars:
    mean = global_sum[v] / global_count[v]
    std = np.sqrt((global_sqsum[v] / global_count[v]) - mean**2)
    std = max(std, 1e-8)
    global_scalers[v] = (mean, std)
    print(f"{v} → mean={mean:.6f}, std={std:.6f}")

# =====================================================
# MAIN LOOP
# =====================================================
for i, file in enumerate(cleaned_files, start=1):
    print(f"\nProcessing Cyclone {i}")
    ds = xr.open_dataset(file)
    
    if "cyclone_name" not in ds.attrs:
        raise ValueError(f"Cyclone {i} missing cyclone_name")
    name = ds.attrs["cyclone_name"]
    
    window = cyclone_windows[i]
    pre, during = split_pre_during(ds, window)
    target_anom = compute_anomaly(pre, during)
    
    # Stack variables
    X = np.stack([pre[v].values for v in input_vars], axis=-1)
    Y = np.stack([target_anom[v].values for v in output_vars], axis=-1)
    
    # Depth selection
    X = X[:, depth_mask, ...]
    Y = Y[:, depth_mask, ...]
    
    # Chilika crop
    X, Y = crop_chilika(X, Y)

    # Scaling
    X_scaled = []
    for idx, v in enumerate(input_vars):
        mean, std = global_scalers[v]
        X_scaled.append((X[..., idx] - mean) / std)
    X = np.stack(X_scaled, axis=-1)

    Y_scaled = []
    for idx, v in enumerate(output_vars):
        mean, std = global_scalers[v]
        Y_scaled.append((Y[..., idx]) / std)
    Y = np.stack(Y_scaled, axis=-1)

    print("X mean:", X.mean(), "std:", X.std())
    print("Y mean:", Y.mean(), "std:", Y.std())

    print(f"{name} X shape: {X.shape}")
    print(f"{name} pre shape: {pre[input_vars[0]].values.shape}")
    print(f"{name} during shape: {during[input_vars[0]].values.shape}")

    # Cyclone forcing (dynamic)
    seq = forcing_sequences[name]  # shape (5,)
    seq_scaled = (seq - wind_mean) / wind_std

    # Forcing shape: (T, D, H, W, 1)
    T, D, H, W, _ = X.shape
    forcing_X = np.zeros((T, D, H, W, 1), dtype=np.float32)
    forcing_Y = np.zeros((T, D, H, W, 1), dtype=np.float32)
    
    # Broadcast forcing along depth, H, W
    forcing_X[:, :, :, :, 0] = seq_scaled[:, None, None, None]  # shape (T,1,1,1)
    forcing_Y[:, :, :, :, 0] = seq_scaled[:, None, None, None]

    # Concatenate forcing channels
    X = np.concatenate([X, forcing_X], axis=-1)
    Y = np.concatenate([Y, forcing_Y], axis=-1)

    # Flatten for Earthformer
    Xef = flatten_with_depth(X, depth_sel)
    Yef = flatten_with_depth(Y, depth_sel)
    
    # Save
    np.save(os.path.join(OUTPUT_DIR, f"{name}_X.npy"), Xef)
    np.save(os.path.join(OUTPUT_DIR, f"{name}_Y.npy"), Yef)
    
    print(f"{name}: X={Xef.shape}, Y={Yef.shape}")

print("\n All cyclones processed correctly.")


Global scalers
to → mean=26.991262, std=2.904903
so → mean=33.453680, std=1.301096
ugo → mean=-0.152653, std=0.241758
vgo → mean=-0.149184, std=0.177203
zo → mean=0.644768, std=0.209564

Processing Cyclone 1
X mean: 0.10299857733742429 std: 0.9941902909463348
Y mean: 0.16182015712326686 std: 0.2091879007024912
Amphan X shape: (5, 18, 8, 8, 5)
Amphan pre shape: (5, 33, 60, 60)
Amphan during shape: (5, 33, 60, 60)
Amphan: X=(5, 8, 8, 126), Y=(5, 8, 8, 90)

Processing Cyclone 2
X mean: 0.024826862411891856 std: 0.6734537705561454
Y mean: 0.08091547517707623 std: 0.13800161880785886
Asani X shape: (5, 18, 8, 8, 5)
Asani pre shape: (5, 33, 60, 60)
Asani during shape: (5, 33, 60, 60)
Asani: X=(5, 8, 8, 126), Y=(5, 8, 8, 90)

Processing Cyclone 3
X mean: -0.0221422530034947 std: 1.132522789458248
Y mean: 0.11081586416697094 std: 0.26030700155355074
Dana X shape: (5, 18, 8, 8, 5)
Dana pre shape: (5, 33, 60, 60)
Dana during shape: (5, 33, 60, 60)
Dana: X=(5, 8, 8, 126), Y=(5, 8, 8, 90)

Proces